# Size-adjusted power experiments (v2)

This notebook is the **new size-adjusted-power driver** for the modified `ci_test.py`.
It is intentionally organized like `run_experiments.ipynb`: run the numbered sections from top to bottom.

The essential change is:

1. run a matching **H0 calibration experiment** for each method;
2. retain the complete H0 p-values;
3. pass those p-values through `null_pvalues=...` when running H1;
4. report `result['size_adjusted_power']`, not raw rejection rates at fixed 0.05/0.10 cutoffs.

Keep this notebook and the modified `ci_test.py` in the same folder. Restart the kernel after replacing `ci_test.py`.


## 0. Setup - import and verify the modified module

This cell deliberately checks the new API. If it fails, Jupyter is still importing an older `ci_test.py`.


In [3]:
%pip install matplotlib pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 46.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 54.4 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 86.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 98.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [matplotlib]9 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [1]:
from copy import deepcopy
import importlib
import inspect
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import ci_test as C
C = importlib.reload(C)

assert hasattr(C, 'size_adjusted_cutoffs'), (
    'The imported ci_test.py is old: size_adjusted_cutoffs is missing.'
)
assert 'null_pvalues' in inspect.signature(C.run_experiment).parameters, (
    'The imported ci_test.py is old: run_experiment has no null_pvalues argument.'
)

print('ci_test loaded from:', Path(C.__file__).resolve())
print('Size-adjusted API check: PASSED')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


ci_test loaded from: /home/23099458d/projects/ci_new/ci_test.py
Size-adjusted API check: PASSED
PyTorch: 2.8.0+cu128
CUDA available: True


## 1. Global experiment settings

Use `RUN_PROFILE='quick'` for an initial check. For final tables, use more H0 replications because the empirical cutoffs themselves must be estimated accurately.


In [2]:
RUN_PROFILE = 'final'      # 'quick' or 'final'

if RUN_PROFILE == 'quick':
    N_REP_ORACLE = 50
    N_REP_H0 = 100
    N_REP_H1 = 100
else:
    N_REP_ORACLE = 200
    N_REP_H0 = 200
    N_REP_H1 = 200

N = 400
LEVELS = (0.10, 0.05)
DGP_NAME = 'skew'
N_JOBS = -1
PREFER_GPU = True

# DGP arguments shared by H0 and H1. Leave empty to use the built-in skew defaults.
BASE_DATA_KWARGS = {}

# Dependence strengths under H1.
ALPHA_GRID = [0.18, 0.20, 0.22,
              0.25, 0.27, 0.30, 0.35, 0.40]

print({
    'profile': RUN_PROFILE,
    'n': N,
    'levels': LEVELS,
    'H0 replications': N_REP_H0,
    'H1 replications per alpha_x': N_REP_H1,
    'DGP': DGP_NAME,
})


{'profile': 'final', 'n': 400, 'levels': (0.1, 0.05), 'H0 replications': 200, 'H1 replications per alpha_x': 200, 'DGP': 'skew'}


In [8]:
RUN_PROFILE = 'final'      # 'quick' or 'final'

if RUN_PROFILE == 'quick':
    N_REP_ORACLE = 50
    N_REP_H0 = 100
    N_REP_H1 = 100
else:
    N_REP_ORACLE = 200
    N_REP_H0 = 200
    N_REP_H1 = 200

N = 400
LEVELS = (0.10, 0.05)
DGP_NAME = 'skew'
N_JOBS = -1
PREFER_GPU = True

# DGP arguments shared by H0 and H1. Leave empty to use the built-in skew defaults.
BASE_DATA_KWARGS = {}

# Dependence strengths under H1.
ALPHA_GRID = [0.15]

print({
    'profile': RUN_PROFILE,
    'n': N,
    'levels': LEVELS,
    'H0 replications': N_REP_H0,
    'H1 replications per alpha_x': N_REP_H1,
    'DGP': DGP_NAME,
})


{'profile': 'final', 'n': 400, 'levels': (0.1, 0.05), 'H0 replications': 200, 'H1 replications per alpha_x': 200, 'DGP': 'skew'}


## 2. Inspect and choose the DGP

The same DGP, sample size, standardization, bootstrap settings, and non-H1 data arguments must be used in the H0 calibration and H1 power stages.


In [9]:
for name, dgp in C.DGPS.items():
    print(f'{name:16s}: {dgp.description}')

assert DGP_NAME in C.DGPS, f'Unknown DGP_NAME={DGP_NAME!r}'


skew            : This project's hard case: skewed lognormal noise, heteroscedastic spread, nonlinear means. Knobs: dx, dy, dz, nstd, dist_z, alpha_x.
gaussian        : Zhang et al. Section 4.1 (Sim 4/5): Z=e3, Y=Z+e1, X=Z+d*e1+(1-d)*e2, d~Bernoulli(alpha_x). Gaussian / homoscedastic / linear. Knobs: dz, alpha_x.
skew_linear     : Simulation 8 (Sec 1.3): the skewed DGP but with LINEAR means m_X=0.8z, m_Y=-0.6z. Knobs: dx, dy, dz, nstd, dist_z, alpha_x.
heteroskedastic : Simulation 5 (Sec 3.1): Y=Z+ey, X=sigma(Z)*ex, sigma(Z)=0.3+1.2|Z|. Gaussian but heteroscedastic with ZERO conditional mean. Knobs: dz, alpha_x.
student_t       : Simulation 5 (Sec 4): heavy-tailed, mX=mY=Z, s(Z)=0.5+|Z|, standardized Student-t noise. Knobs: dz, alpha_x, df (default 3).


## 3. Define the two methods

Two designs are available:

- `separately_tuned`: compare the two complete parameter sets currently used in the experiments;
- `strict_ablation`: keep every setting identical and change only the alignment mechanism.

For a causal statement about the effect of alignment itself, use `strict_ablation`.


In [10]:
COMPARISON_MODE = 'separately_tuned'   # 'separately_tuned' or 'strict_ablation'

# -----------------------------------------------------------------------------
# Current tuned configuration: no alignment (lambda=0)
# -----------------------------------------------------------------------------
CONFIG_L0_TUNED = dict(C.DEFAULT_CONFIG)
CONFIG_L0_TUNED.update(
    depth=2,
    width=1024,
    noise_dim=5,
    dropout=0.0,

    lr=5e-4,
    epochs=600,
    batch_size=128,
    grad_clip=None,
    weight_decay=1e-5,
    M_train=50,
    mmd_w_laplacian=1.0,
    mmd_w_gaussian=1.0,

    early_stop=True,
    min_epochs=100,
    patience=80,
    min_delta=1e-5,
    lr_scheduler=True,
    lr_factor=0.4,
    lr_patience=15,
    min_lr_frac=0.05,

    align_mode='none',
    lambda_align=0.0,
    taus=(0.1, 0.5, 0.9),
    align_samples=64,

    n_folds=2,
    M_test=100,
    n_boot=1000,
    boot_rv='gaussian',
    standardize=True,
)

# -----------------------------------------------------------------------------
# Current tuned configuration: multi-quantile alignment (lambda=0.5)
# -----------------------------------------------------------------------------
CONFIG_L05_TUNED = dict(C.DEFAULT_CONFIG)
CONFIG_L05_TUNED.update(
    depth=2,
    width=1024,
    noise_dim=5,
    dropout=0.0,

    lr=5e-3,
    epochs=600,
    batch_size=128,
    grad_clip=None,
    weight_decay=1e-5,
    M_train=30,
    mmd_w_laplacian=1.0,
    mmd_w_gaussian=1.0,

    early_stop=True,
    min_epochs=100,
    patience=80,
    min_delta=1e-4,
    lr_scheduler=True,
    lr_factor=0.1,
    lr_patience=15,
    min_lr_frac=0.05,

    align_mode='quantile',
    lambda_align=0.5,
    taus=(0.1, 0.5, 0.9),
    align_samples=64,

    n_folds=2,
    M_test=100,
    n_boot=1000,
    boot_rv='gaussian',
    standardize=True,
)

# Strict ablation: both methods share CONFIG_L0_TUNED except for alignment.
STRICT_BASE = deepcopy(CONFIG_L0_TUNED)

CONFIG_L0_STRICT = deepcopy(STRICT_BASE)
CONFIG_L0_STRICT.update(align_mode='none', lambda_align=0.0)

CONFIG_L05_STRICT = deepcopy(STRICT_BASE)
CONFIG_L05_STRICT.update(
    align_mode='quantile',
    lambda_align=0.5,
    taus=(0.1, 0.5, 0.9),
    align_samples=64,
)

if COMPARISON_MODE == 'separately_tuned':
    METHODS = {
        'lambda_0': CONFIG_L0_TUNED,
        'lambda_05': CONFIG_L05_TUNED,
    }
elif COMPARISON_MODE == 'strict_ablation':
    METHODS = {
        'lambda_0': CONFIG_L0_STRICT,
        'lambda_05': CONFIG_L05_STRICT,
    }
else:
    raise ValueError("COMPARISON_MODE must be 'separately_tuned' or 'strict_ablation'.")

METHOD_LABELS = {
    'lambda_0': 'No alignment (lambda=0)',
    'lambda_05': 'Quantile alignment (lambda=0.5)',
}

print('Comparison mode:', COMPARISON_MODE)
for method_id in METHODS:
    print(method_id, '->', METHOD_LABELS[method_id])


Comparison mode: separately_tuned
lambda_0 -> No alignment (lambda=0)
lambda_05 -> Quantile alignment (lambda=0.5)


### 3.1 Verify which parameters differ

In `strict_ablation` mode, only `align_mode` and `lambda_align` should differ. This cell prevents accidental comparison of mislabeled configurations.


In [14]:
def config_difference_table(config_a, config_b):
    keys = sorted(set(config_a) | set(config_b))
    rows = []
    for key in keys:
        value_a = config_a.get(key, '<missing>')
        value_b = config_b.get(key, '<missing>')
        if value_a != value_b:
            rows.append({
                'parameter': key,
                'lambda_0': value_a,
                'lambda_05': value_b,
            })
    return pd.DataFrame(rows)

CONFIG_DIFFERENCES = config_difference_table(METHODS['lambda_0'], METHODS['lambda_05'])
display(CONFIG_DIFFERENCES)

if COMPARISON_MODE == 'strict_ablation':
    allowed = {'align_mode', 'lambda_align'}
    unexpected = set(CONFIG_DIFFERENCES['parameter']) - allowed
    assert not unexpected, f'Unexpected differences in strict ablation: {sorted(unexpected)}'


,parameter,lambda_0,lambda_05
0,M_train,50,30
1,align_mode,none,quantile
2,lambda_align,0.0,0.5
3,lr,0.0005,0.005
4,lr_factor,0.4,0.1
5,min_delta,0.00001,0.0001


## 4. Oracle H0 sanity check

Run this before training learned generators. Oracle size should be near the nominal levels. If oracle is abnormal, inspect the statistic/bootstrap before tuning neural-network parameters.


In [ ]:
ORACLE_RESULT = C.run_experiment(
    n=N,
    hypothesis='H0',
    n_rep=N_REP_ORACLE,
    config=METHODS['lambda_0'],
    dgp=DGP_NAME,
    oracle=True,
    levels=LEVELS,
    data_kwargs=BASE_DATA_KWARGS,
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
)
ORACLE_RESULT


## 5. Run matching H0 calibration experiments

This is the required first stage for size-adjusted power. Each method receives its **own** H0 p-values obtained using the same method configuration, DGP, sample size, and non-H1 data settings.


In [10]:
H0_RESULTS = {}

for method_id, config in METHODS.items():
    print()
    print('=' * 88)
    print('H0 calibration:', METHOD_LABELS[method_id])
    print('=' * 88)

    H0_RESULTS[method_id] = C.run_experiment(
        n=N,
        hypothesis='H0',
        n_rep=N_REP_H0,
        config=config,
        dgp=DGP_NAME,
        oracle=False,
        levels=LEVELS,
        data_kwargs=BASE_DATA_KWARGS,
        n_jobs=N_JOBS,
        prefer_gpu=PREFER_GPU,
        verbose=True,
    )



H0 calibration: No alignment (lambda=0)
[info] CUDA GPU detected -> running on GPU with n_jobs=1 (ignoring n_jobs=-1; the GPU already accelerates each replicate).
[H0 dgp=skew depth=2 n=400 reps=200] rej@0.10=0.160  rej@0.05=0.095

H0 calibration: Quantile alignment (lambda=0.5)
[info] CUDA GPU detected -> running on GPU with n_jobs=1 (ignoring n_jobs=-1; the GPU already accelerates each replicate).
[H0 dgp=skew depth=2 n=400 reps=200] rej@0.10=0.115  rej@0.05=0.040


## 6. Inspect empirical Type I error and adjusted cutoffs

The modified `ci_test.py` converts each method's H0 p-values into empirical level-specific cutoffs. Those cutoffs, rather than fixed 0.05 and 0.10, are used in H1.


In [11]:
calibration_rows = []
ADJUSTED_CUTOFFS = {}

for method_id, h0_result in H0_RESULTS.items():
    p0 = np.asarray(h0_result['pvalues'])
    cutoffs = C.size_adjusted_cutoffs(h0_result, levels=LEVELS)
    ADJUSTED_CUTOFFS[method_id] = cutoffs

    for level in LEVELS:
        cutoff = cutoffs[float(level)]
        calibration_rows.append({
            'method_id': method_id,
            'method': METHOD_LABELS[method_id],
            'nominal_level': float(level),
            'raw_H0_rejection': h0_result['rejection'][level],
            'adjusted_cutoff': cutoff,
            'H0_rejection_at_adjusted_cutoff': float(np.mean(p0 <= cutoff)),
            'H0_replications': len(p0),
        })

CALIBRATION_TABLE = pd.DataFrame(calibration_rows)
display(CALIBRATION_TABLE)


,method_id,method,nominal_level,raw_H0_rejection,adjusted_cutoff,H0_rejection_at_adjusted_cutoff,H0_replications
0,lambda_0,No alignment (lambda=0),0.10,0.160,0.050,0.100,200
1,lambda_0,No alignment (lambda=0),0.05,0.095,0.026,0.055,200
2,lambda_05,Quantile alignment (lambda=0.5),0.10,0.115,0.087,0.100,200
3,lambda_05,Quantile alignment (lambda=0.5),0.05,0.040,0.055,0.050,200


### 6.1 Save H0 calibration p-values

H0 calibration is expensive. Save it before running the H1 grid. After a kernel restart, you may reload these p-values instead of repeating H0.


In [12]:
H0_FILE = Path(f'h0_calibration_{DGP_NAME}_{COMPARISON_MODE}_v2.npz')

np.savez_compressed(
    H0_FILE,
    **{
        method_id: np.asarray(result['pvalues'])
        for method_id, result in H0_RESULTS.items()
    },
)

CALIBRATION_TABLE.to_csv(
    f'h0_calibration_summary_{DGP_NAME}_{COMPARISON_MODE}_v2.csv',
    index=False,
)

print('Saved H0 p-values to:', H0_FILE.resolve())


Saved H0 p-values to: /home/23099458d/projects/ci_new/h0_calibration_skew_separately_tuned_v2.npz


### 6.2 Optional: reload previously saved H0 p-values

Run this only after a kernel restart when you want to skip Section 5.


In [11]:
from pathlib import Path
import numpy as np

# 优先检查 notebook 主目录中保存的 H0 文件
H0_FILE = Path(
    f"h0_calibration_{DGP_NAME}_{COMPARISON_MODE}_v2.npz"
)

# 如果主目录没有，则检查结果文件夹
if not H0_FILE.exists():
    H0_FILE = Path(
        f"size_adjusted_power_results_{COMPARISON_MODE}_v2"
    ) / "h0_pvalues.npz"

if not H0_FILE.exists():
    raise FileNotFoundError(
        "没有找到保存的 H0 p-values。"
        "如果之前只记录了 rej rate，而没有保存完整 p-values，"
        "则需要重新运行一次 H0 calibration。"
    )

loaded = np.load(H0_FILE)

H0_RESULTS = {}

for method_id in METHODS:
    if method_id not in loaded.files:
        raise KeyError(
            f"H0 文件中没有方法 {method_id!r}。"
            f"文件中已有的键为：{loaded.files}"
        )

    p0 = np.asarray(loaded[method_id], dtype=float)

    H0_RESULTS[method_id] = {
        "pvalues": p0,
        "rejection": {
            float(level): float(np.mean(p0 < float(level)))
            for level in LEVELS
        },
    }

    cutoffs = C.size_adjusted_cutoffs(
        p0,
        levels=LEVELS
    )

    print(
        f"{method_id}: "
        f"H0 reps={len(p0)}, "
        f"raw size={H0_RESULTS[method_id]['rejection']}, "
        f"adjusted cutoffs={cutoffs}"
    )

print("Loaded H0 p-values from:", H0_FILE.resolve())

lambda_0: H0 reps=200, raw size={0.1: 0.16, 0.05: 0.095}, adjusted cutoffs={0.1: 0.05, 0.05: 0.026}
lambda_05: H0 reps=200, raw size={0.1: 0.115, 0.05: 0.04}, adjusted cutoffs={0.1: 0.087, 0.05: 0.055}
Loaded H0 p-values from: /home/23099458d/projects/ci_new/h0_calibration_skew_separately_tuned_v2.npz


In [12]:
# Uncomment when needed.
loaded = np.load(H0_FILE)
H0_RESULTS = {}
for method_id in METHODS:
     p0 = np.asarray(loaded[method_id])
     H0_RESULTS[method_id] = {
         'pvalues': p0,
         'rejection': {
             level: float(np.mean(p0 < level))
             for level in LEVELS
         },
     }
print('Reloaded H0 calibration for:', list(H0_RESULTS))


Reloaded H0 calibration for: ['lambda_0', 'lambda_05']


## 7. Run H1 size-adjusted power

The key argument is:

```python
null_pvalues=H0_RESULTS[method_id]
```

The modified `ci_test.py` will raise an error if H1 is requested without matching H0 p-values. The reported values below are size-adjusted power.


In [7]:
POWER_ROWS = []
H1_RESULTS = {method_id: {} for method_id in METHODS}

for alpha_x in ALPHA_GRID:
    print()
    print('-' * 88)
    print(f'alpha_x = {alpha_x:.2f}')
    print('-' * 88)

    for method_id, config in METHODS.items():
        h1_data_kwargs = dict(BASE_DATA_KWARGS)
        h1_data_kwargs['alpha_x'] = float(alpha_x)

        result = C.run_experiment(
            n=N,
            hypothesis='H1',
            n_rep=N_REP_H1,
            config=config,
            dgp=DGP_NAME,
            oracle=False,
            levels=LEVELS,
            data_kwargs=h1_data_kwargs,
            n_jobs=N_JOBS,
            prefer_gpu=PREFER_GPU,
            verbose=False,
            null_pvalues=H0_RESULTS[method_id],
        )

        H1_RESULTS[method_id][float(alpha_x)] = result

        row = {
            'method_id': method_id,
            'method': METHOD_LABELS[method_id],
            'alpha_x': float(alpha_x),
            'size_adjusted_power_0.10': result['size_adjusted_power'][0.10],
            'size_adjusted_power_0.05': result['size_adjusted_power'][0.05],
            'cutoff_0.10': result['cutoffs'][0.10],
            'cutoff_0.05': result['cutoffs'][0.05],
            'H1_replications': len(result['pvalues']),
        }
        POWER_ROWS.append(row)

        print(
            f"{METHOD_LABELS[method_id]:36s}  "
            f"adjusted power@0.10={row['size_adjusted_power_0.10']:.3f}  "
            f"adjusted power@0.05={row['size_adjusted_power_0.05']:.3f}"
        )

POWER_TABLE = pd.DataFrame(POWER_ROWS)
display(POWER_TABLE)



----------------------------------------------------------------------------------------
alpha_x = 0.18
----------------------------------------------------------------------------------------
No alignment (lambda=0)               adjusted power@0.10=0.355  adjusted power@0.05=0.235
Quantile alignment (lambda=0.5)       adjusted power@0.10=0.525  adjusted power@0.05=0.400

----------------------------------------------------------------------------------------
alpha_x = 0.20
----------------------------------------------------------------------------------------
No alignment (lambda=0)               adjusted power@0.10=0.440  adjusted power@0.05=0.320
Quantile alignment (lambda=0.5)       adjusted power@0.10=0.600  adjusted power@0.05=0.530

----------------------------------------------------------------------------------------
alpha_x = 0.22
----------------------------------------------------------------------------------------
No alignment (lambda=0)               adjusted power@0

,method_id,method,alpha_x,size_adjusted_power_0.10,size_adjusted_power_0.05,cutoff_0.10,cutoff_0.05,H1_replications
0,lambda_0,No alignment (lambda=0),0.18,0.355,0.235,0.050,0.026,200
1,lambda_05,Quantile alignment (lambda=0.5),0.18,0.525,0.400,0.087,0.055,200
2,lambda_0,No alignment (lambda=0),0.20,0.440,0.320,0.050,0.026,200
3,lambda_05,Quantile alignment (lambda=0.5),0.20,0.600,0.530,0.087,0.055,200
4,lambda_0,No alignment (lambda=0),0.22,0.535,0.405,0.050,0.026,200
5,lambda_05,Quantile alignment (lambda=0.5),0.22,0.705,0.575,0.087,0.055,200
6,lambda_0,No alignment (lambda=0),0.25,0.695,0.555,0.050,0.026,200
7,lambda_05,Quantile alignment (lambda=0.5),0.25,0.830,0.745,0.087,0.055,200
8,lambda_0,No alignment (lambda=0),0.27,0.775,0.645,0.050,0.026,200
9,lambda_05,Quantile alignment (lambda=0.5),0.27,0.865,0.815,0.087,0.055,200


In [13]:
POWER_ROWS = []
H1_RESULTS = {method_id: {} for method_id in METHODS}

for alpha_x in ALPHA_GRID:
    print()
    print('-' * 88)
    print(f'alpha_x = {alpha_x:.2f}')
    print('-' * 88)

    for method_id, config in METHODS.items():
        h1_data_kwargs = dict(BASE_DATA_KWARGS)
        h1_data_kwargs['alpha_x'] = float(alpha_x)

        result = C.run_experiment(
            n=N,
            hypothesis='H1',
            n_rep=N_REP_H1,
            config=config,
            dgp=DGP_NAME,
            oracle=False,
            levels=LEVELS,
            data_kwargs=h1_data_kwargs,
            n_jobs=N_JOBS,
            prefer_gpu=PREFER_GPU,
            verbose=False,
            null_pvalues=H0_RESULTS[method_id],
        )

        H1_RESULTS[method_id][float(alpha_x)] = result

        row = {
            'method_id': method_id,
            'method': METHOD_LABELS[method_id],
            'alpha_x': float(alpha_x),
            'size_adjusted_power_0.10': result['size_adjusted_power'][0.10],
            'size_adjusted_power_0.05': result['size_adjusted_power'][0.05],
            'cutoff_0.10': result['cutoffs'][0.10],
            'cutoff_0.05': result['cutoffs'][0.05],
            'H1_replications': len(result['pvalues']),
        }
        POWER_ROWS.append(row)

        print(
            f"{METHOD_LABELS[method_id]:36s}  "
            f"adjusted power@0.10={row['size_adjusted_power_0.10']:.3f}  "
            f"adjusted power@0.05={row['size_adjusted_power_0.05']:.3f}"
        )

POWER_TABLE = pd.DataFrame(POWER_ROWS)
display(POWER_TABLE)



----------------------------------------------------------------------------------------
alpha_x = 0.15
----------------------------------------------------------------------------------------
No alignment (lambda=0)               adjusted power@0.10=0.250  adjusted power@0.05=0.155
Quantile alignment (lambda=0.5)       adjusted power@0.10=0.370  adjusted power@0.05=0.290


,method_id,method,alpha_x,size_adjusted_power_0.10,size_adjusted_power_0.05,cutoff_0.10,cutoff_0.05,H1_replications
0,lambda_0,No alignment (lambda=0),0.15,0.25,0.155,0.050,0.026,200
1,lambda_05,Quantile alignment (lambda=0.5),0.15,0.37,0.290,0.087,0.055,200


## 8. Comparison tables

These tables compare methods at the same empirical size and are the primary results to report.


In [ ]:
POWER_COMPARISON_005 = POWER_TABLE.pivot(
    index='alpha_x',
    columns='method',
    values='size_adjusted_power_0.05',
).sort_index()

POWER_COMPARISON_010 = POWER_TABLE.pivot(
    index='alpha_x',
    columns='method',
    values='size_adjusted_power_0.10',
).sort_index()

print('Size-adjusted power at empirical size 0.05')
display(POWER_COMPARISON_005)

print('Size-adjusted power at empirical size 0.10')
display(POWER_COMPARISON_010)


## 9. Power curves

In [ ]:
for level, value_column in [
    (0.05, 'size_adjusted_power_0.05'),
    (0.10, 'size_adjusted_power_0.10'),
]:
    plt.figure(figsize=(7.5, 5.0))

    for method_id in METHODS:
        subset = POWER_TABLE[
            POWER_TABLE['method_id'] == method_id
        ].sort_values('alpha_x')

        plt.plot(
            subset['alpha_x'],
            subset[value_column],
            marker='o',
            label=METHOD_LABELS[method_id],
        )

    plt.xlabel('Dependence strength alpha_x')
    plt.ylabel('Size-adjusted power')
    plt.title(f'Size-adjusted power at empirical size {level:.2f}')
    plt.ylim(0.0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 10. Save results

In [ ]:
OUTPUT_DIR = Path(f'size_adjusted_power_results_{COMPARISON_MODE}_v2')
OUTPUT_DIR.mkdir(exist_ok=True)

POWER_TABLE.to_csv(OUTPUT_DIR / 'size_adjusted_power_long.csv', index=False)
POWER_COMPARISON_005.to_csv(OUTPUT_DIR / 'size_adjusted_power_at_0.05.csv')
POWER_COMPARISON_010.to_csv(OUTPUT_DIR / 'size_adjusted_power_at_0.10.csv')
CALIBRATION_TABLE.to_csv(OUTPUT_DIR / 'h0_calibration_summary.csv', index=False)

np.savez_compressed(
    OUTPUT_DIR / 'h0_pvalues.npz',
    **{
        method_id: np.asarray(result['pvalues'])
        for method_id, result in H0_RESULTS.items()
    },
)

h1_arrays = {}
for method_id, by_alpha in H1_RESULTS.items():
    for alpha_x, result in by_alpha.items():
        alpha_tag = f'{alpha_x:.2f}'.replace('.', 'p')
        h1_arrays[f'{method_id}_alpha_{alpha_tag}'] = np.asarray(result['pvalues'])
np.savez_compressed(OUTPUT_DIR / 'h1_pvalues.npz', **h1_arrays)

print('Saved results to:', OUTPUT_DIR.resolve())


## 11. Minimal single-method template

This is the shortest correct workflow for one method: H0 calibration first, then H1 with `null_pvalues=H0_RESULT`.


In [ ]:
# Example: only the quantile-alignment method.
ACTIVE_CONFIG = METHODS['lambda_05']

# Stage 1: matching H0 calibration
ACTIVE_H0 = C.run_experiment(
    n=N,
    hypothesis='H0',
    n_rep=N_REP_H0,
    config=ACTIVE_CONFIG,
    dgp=DGP_NAME,
    oracle=False,
    levels=LEVELS,
    data_kwargs=BASE_DATA_KWARGS,
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
)

# Stage 2: size-adjusted H1 power
ACTIVE_H1 = C.run_experiment(
    n=N,
    hypothesis='H1',
    n_rep=N_REP_H1,
    config=ACTIVE_CONFIG,
    dgp=DGP_NAME,
    oracle=False,
    levels=LEVELS,
    data_kwargs={**BASE_DATA_KWARGS, 'alpha_x': 0.20},
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
    null_pvalues=ACTIVE_H0,
)

print('Size-adjusted power:', ACTIVE_H1['size_adjusted_power'])
print('Adjusted cutoffs:', ACTIVE_H1['cutoffs'])


In [ ]:
----------------------------------------------------------------------------------------
alpha_x = 0.10
----------------------------------------------------------------------------------------
No alignment (lambda=0)               adjusted power@0.10=0.130  adjusted power@0.05=0.075
Quantile alignment (lambda=0.5)       adjusted power@0.10=0.240  adjusted power@0.05=0.155

----------------------------------------------------------------------------------------
alpha_x = 0.15
----------------------------------------------------------------------------------------
No alignment (lambda=0)               adjusted power@0.10=0.250  adjusted power@0.05=0.155

----------------------------------------------------------------------------------------
alpha_x = 0.18
----------------------------------------------------------------------------------------
No alignment (lambda=0)               adjusted power@0.10=0.355  adjusted power@0.05=0.235
Quantile alignment (lambda=0.5)       adjusted power@0.10=0.525  adjusted power@0.05=0.400

----------------------------------------------------------------------------------------
alpha_x = 0.20
----------------------------------------------------------------------------------------
No alignment (lambda=0)               adjusted power@0.10=0.440  adjusted power@0.05=0.320
Quantile alignment (lambda=0.5)       adjusted power@0.10=0.600  adjusted power@0.05=0.530

----------------------------------------------------------------------------------------
alpha_x = 0.22
----------------------------------------------------------------------------------------
No alignment (lambda=0)               adjusted power@0.10=0.535  adjusted power@0.05=0.405
Quantile alignment (lambda=0.5)       adjusted power@0.10=0.705  adjusted power@0.05=0.575

----------------------------------------------------------------------------------------
alpha_x = 0.25
----------------------------------------------------------------------------------------
No alignment (lambda=0)               adjusted power@0.10=0.695  adjusted power@0.05=0.555
Quantile alignment (lambda=0.5)       adjusted power@0.10=0.830  adjusted power@0.05=0.745

----------------------------------------------------------------------------------------
alpha_x = 0.27
----------------------------------------------------------------------------------------
No alignment (lambda=0)               adjusted power@0.10=0.775  adjusted power@0.05=0.645
Quantile alignment (lambda=0.5)       adjusted power@0.10=0.865  adjusted power@0.05=0.815

---------------------------------------------------------------------------------------